# Eksperimen 21A (EXP21A): Anti-Leakage Preprocessing & Clean Attribute Pipeline

Notebook ini mengimplementasikan eksperimen `EXP21A`:
- Menjawab saran pembimbing mengenai hasil 'terlalu sempurna' (Recall 100%, F1 0.94) di EXP21.
- Mengeliminasi kebocoran atribut quasi-identifier (`Zip` dan `Merchant City` dihapus dari input XGBoost).
- Menerapkan *Strict Out-of-Sample Preprocessing*: Scaler dan Dictionary Kategori HANYA di-fit pada Train Window.
- Regularisasi pohon XGBoost (`max_depth=4`, `min_child_weight=10`, `scale_pos_weight=sqrt(ratio)`).
- Audit *Feature Importance* komparatif: membandingkan bobot keputusan Tabular vs 128-d GraphSAGE.


In [ ]:
%%writefile config.kaggle.yaml
data:
  transactions: "/kaggle/input/credit-card-transactions-ibm-v2/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016

time_window:
  reference_date: "2018-01-01"
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60

features:
  mode: "clean"
  use_cyclical_time: true
  exclude_high_cardinality: true

sampling:
  method: "uniform"
  undersampling_ratio: 1.0

model:
  hidden_channels: 128
  out_channels: 1
  num_layers: 2
  aggr: "mean"
  dropout: 0.2

training:
  batch_size: 1024
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  label_smoothing: 0.0
  grad_clip_norm: 5.0
  warmup_epochs: 0
  use_lr_scheduler: true
  lr_scheduler_factor: 0.5
  lr_scheduler_patience: 3
  early_stopping_patience: 7
  loss_weight_strategy: "sqrt"
  max_pos_weight: 50.0
  log_step_interval: 1
  num_neighbors: [15, 10]

classifier:
  type: "xgboost"
  n_estimators: 100
  max_depth: 4
  learning_rate: 0.05
  min_child_weight: 10
  subsample: 0.8
  colsample_bytree: 0.8
  scale_pos_weight_strategy: "sqrt"

output:
  model_dir: "/kaggle/working/model/exp21a"
  result_dir: "/kaggle/working/result/exp21a"



In [ ]:
%%writefile config.py
import yaml
from pathlib import Path

def load_config(config_path: str = "config.reference.yaml") -> dict:
    """
    Memuat file konfigurasi YAML untuk EXP21A.
    """
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"File konfigurasi tidak ditemukan: {config_path}")
    
    with open(path, "r") as f:
        config = yaml.safe_load(f)
        
    return config



In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler
from pathlib import Path

def load_and_split_raw_data(
    file_path: str, 
    start_year: int = 2016, 
    nrows: int = None,
    reference_date: str = '2018-01-01',
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60
) -> tuple:
    """
    Memuat dataset mentah IBM Credit Card Transactions, memfilter era modern (Year >= start_year),
    melakukan parsing datetime kronologis, dan memotongnya ke dalam 4 partisi temporal SHINE FRAUD II
    SEBELUM proses rekayasa fitur atau penskalaan (Strict Out-of-Sample Protocol).
    """
    print(f"Loading data from {file_path} (Filter: Year >= {start_year}, nrows={nrows})...")
    
    # 1. Pemuatan chunked efisien jika start_year ditentukan
    if start_year is not None:
        chunks = []
        collected_rows = 0
        chunksize = 1_000_000
        
        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            filtered_chunk = chunk[chunk['Year'] >= start_year]
            if len(filtered_chunk) > 0:
                if nrows is not None:
                    remaining = nrows - collected_rows
                    if remaining <= 0:
                        break
                    filtered_chunk = filtered_chunk.iloc[:remaining]
                chunks.append(filtered_chunk)
                collected_rows += len(filtered_chunk)
                if nrows is not None and collected_rows >= nrows:
                    break
                    
        if not chunks:
            raise ValueError(f"Tidak ada baris yang ditemukan dengan Year >= {start_year} pada file {file_path}")
            
        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(file_path, nrows=nrows)

    print(f"Memproses {len(df)} baris data mentah...")
    print("Parsing timestamps...")
    df['Time'] = df['Time'].fillna('00:00')
    df['datetime'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + 
        df['Month'].astype(str).str.zfill(2) + '-' + 
        df['Day'].astype(str).str.zfill(2) + ' ' + 
        df['Time']
    )
    
    print("Sorting chronologically...")
    df = df.sort_values('datetime').reset_index(drop=True)
    
    print("Cleaning Amount & Target Label...")
    df['Amount'] = df['Amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})
    
    # 2. Pembagian Jendela Waktu Temporal (SHINE FRAUD II)
    print("\nMelakukan pembagian temporal window strictly chronologic...")
    t0 = pd.to_datetime(reference_date)
    t_val_start = t0 + timedelta(days=train_days)
    t_gap_start = t_val_start + timedelta(days=val_days)
    t_test_start = t_gap_start + timedelta(days=gap_days)
    t_test_end = t_test_start + timedelta(days=test_days)
    
    train_df = df[(df['datetime'] >= t0) & (df['datetime'] < t_val_start)].copy().reset_index(drop=True)
    val_df = df[(df['datetime'] >= t_val_start) & (df['datetime'] < t_gap_start)].copy().reset_index(drop=True)
    gap_df = df[(df['datetime'] >= t_gap_start) & (df['datetime'] < t_test_start)].copy().reset_index(drop=True)
    test_df = df[(df['datetime'] >= t_test_start) & (df['datetime'] < t_test_end)].copy().reset_index(drop=True)
    
    def compute_split_stats(name: str, split_df: pd.DataFrame, start_dt, end_dt):
        n_tx = len(split_df)
        if n_tx == 0:
            return {
                "name": name,
                "start_date": str(start_dt),
                "end_date": str(end_dt),
                "num_transactions": 0,
                "num_fraud": 0,
                "fraud_rate_pct": 0.0,
                "unique_users": 0,
                "unique_merchants": 0,
                "total_nodes": 0
            }
        n_fraud = int(split_df['Is Fraud?'].sum())
        n_users = int(split_df['User'].nunique())
        n_merchants = int(split_df['Merchant Name'].nunique())
        return {
            "name": name,
            "start_date": str(start_dt),
            "end_date": str(end_dt),
            "num_transactions": n_tx,
            "num_fraud": n_fraud,
            "fraud_rate_pct": round((n_fraud / n_tx) * 100, 4),
            "unique_users": n_users,
            "unique_merchants": n_merchants,
            "total_nodes": n_tx + n_users + n_merchants
        }

    total_window_df = pd.concat([train_df, val_df, gap_df, test_df], ignore_index=True)
    
    stats_dict = {
        "protocol": "SHINE FRAUD II Rolling Window Alignment (EXP21A Anti-Leakage)",
        "reference_date_T0": str(t0),
        "durations_days": {
            "train": train_days,
            "validation": val_days,
            "embargo_gap": gap_days,
            "test": test_days,
            "total_horizon": train_days + val_days + gap_days + test_days
        },
        "splits": {
            "train": compute_split_stats("Train", train_df, t0, t_val_start),
            "validation": compute_split_stats("Validation", val_df, t_val_start, t_gap_start),
            "embargo_gap": compute_split_stats("Embargo Gap", gap_df, t_gap_start, t_test_start),
            "test": compute_split_stats("Test", test_df, t_test_start, t_test_end),
            "total_window": compute_split_stats("Total Window", total_window_df, t0, t_test_end)
        }
    }
    
    return train_df, val_df, test_df, gap_df, stats_dict

def apply_strict_anti_leakage_preprocessing(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    gap_df: pd.DataFrame = None,
    feature_mode: str = "clean",
    use_cyclical_time: bool = True
) -> tuple:
    """
    Menerapkan Strict Anti-Leakage Preprocessing Protocol:
    1. StandardScaler Amount HANYA di-fit pada train_df, lalu mentransformasikan val_df dan test_df.
    2. Encoding kategorikal HANYA mempelajari kamus nilai unik dari train_df. Nilai baru pada val/test dipetakan ke 0 (<oov>).
    3. Mengeliminasi Zip dan Merchant City (mode: clean) untuk mencegah pohon keputusan menghafal ID kuasi.
    4. Menambahkan fitur waktu siklik (sin/cos jam & hari) yang tidak rentan memorization.
    """
    print(f"\n[STRICT ANTI-LEAKAGE] Menerapkan preprocessing (Mode: {feature_mode}, Cyclical Time: {use_cyclical_time})...")
    
    # Tentukan atribut kategorikal yang diizinkan
    if feature_mode == "clean":
        cat_columns = ['Use Chip', 'Merchant State', 'MCC', 'Errors?']
        print(f"  • Fitur High-Cardinality ('Zip', 'Merchant City') DIHAPUS dari input!")
    elif feature_mode == "clean_basic":
        cat_columns = ['Use Chip', 'MCC', 'Errors?']
        print(f"  • Menggunakan fitur kategorikal minimal: {cat_columns}")
    elif feature_mode == "legacy_leaky":
        cat_columns = ['Use Chip', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?']
        print(f"  • [WARNING] Mode legacy_leaky: Termasuk 'Zip' dan 'Merchant City'.")
    else:
        raise ValueError(f"Feature mode '{feature_mode}' tidak dikenal. Pilih 'clean', 'clean_basic', atau 'legacy_leaky'.")

    # A. Standarisasi Fitur Numerik (Amount) — FIT HANYA DI TRAIN
    scaler_amount = StandardScaler()
    train_df['Amount'] = scaler_amount.fit_transform(train_df[['Amount']])
    val_df['Amount'] = scaler_amount.transform(val_df[['Amount']])
    test_df['Amount'] = scaler_amount.transform(test_df[['Amount']])
    if gap_df is not None and len(gap_df) > 0:
        gap_df['Amount'] = scaler_amount.transform(gap_df[['Amount']])
        
    print(f"  • Amount Scaler fit pada Train: mean={scaler_amount.mean_[0]:.4f}, std={scaler_amount.scale_[0]:.4f}")

    # B. Robust Categorical Encoding — DICTIONARY FIT HANYA DI TRAIN
    categorical_encoders = {}
    for col in cat_columns:
        train_df[col] = train_df[col].fillna('<missing>').astype(str)
        val_df[col] = val_df[col].fillna('<missing>').astype(str)
        test_df[col] = test_df[col].fillna('<missing>').astype(str)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].fillna('<missing>').astype(str)
            
        # Pelajari mapping unik dari TRAIN saja
        unique_train_vals = sorted(train_df[col].unique().tolist())
        # Index 0 dicadangkan untuk OOV (<unknown>), kategori train mulai dari index 1
        cat_to_id = {val: idx + 1 for idx, val in enumerate(unique_train_vals)}
        categorical_encoders[col] = cat_to_id
        
        # Transform dengan fallback OOV = 0
        train_df[col] = train_df[col].map(cat_to_id).fillna(0).astype(int)
        val_df[col] = val_df[col].map(cat_to_id).fillna(0).astype(int)
        test_df[col] = test_df[col].map(cat_to_id).fillna(0).astype(int)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].map(cat_to_id).fillna(0).astype(int)
            
        num_oov_val = int((val_df[col] == 0).sum())
        num_oov_test = int((test_df[col] == 0).sum())
        print(f"  • Kolom '{col}': {len(cat_to_id)} kategori train dipelajari (OOV di Val: {num_oov_val}, di Test: {num_oov_test})")

    # C. Fitur Temporal Siklik (Cyclical Time Features)
    feature_cols = ['Amount'] + cat_columns
    if use_cyclical_time:
        for split_df in [train_df, val_df, test_df]:
            hour_float = split_df['datetime'].dt.hour + split_df['datetime'].dt.minute / 60.0
            split_df['hour_sin'] = np.sin(2 * np.pi * hour_float / 24.0).astype(np.float32)
            split_df['hour_cos'] = np.cos(2 * np.pi * hour_float / 24.0).astype(np.float32)
            
            dow = split_df['datetime'].dt.dayofweek
            split_df['dow_sin'] = np.sin(2 * np.pi * dow / 7.0).astype(np.float32)
            split_df['dow_cos'] = np.cos(2 * np.pi * dow / 7.0).astype(np.float32)
            
        if gap_df is not None and len(gap_df) > 0:
            hour_float = gap_df['datetime'].dt.hour + gap_df['datetime'].dt.minute / 60.0
            gap_df['hour_sin'] = np.sin(2 * np.pi * hour_float / 24.0).astype(np.float32)
            gap_df['hour_cos'] = np.cos(2 * np.pi * hour_float / 24.0).astype(np.float32)
            dow = gap_df['datetime'].dt.dayofweek
            gap_df['dow_sin'] = np.sin(2 * np.pi * dow / 7.0).astype(np.float32)
            gap_df['dow_cos'] = np.cos(2 * np.pi * dow / 7.0).astype(np.float32)

        time_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
        feature_cols = feature_cols + time_cols
        print(f"  • Ditambahkan 4 fitur waktu siklik: {time_cols}")

    print(f"Total fitur transaksi bersih (feature_cols): {len(feature_cols)} fitur -> {feature_cols}")
    
    artifacts = {
        "scaler_amount": scaler_amount,
        "categorical_encoders": categorical_encoders,
        "feature_cols": feature_cols,
        "feature_mode": feature_mode
    }
    
    return train_df, val_df, test_df, gap_df, feature_cols, artifacts



In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler

def graph_level_undersample(df: pd.DataFrame, ratio: float = 1.0, seed: int = 42) -> pd.DataFrame:
    """
    Melakukan Graph-Level Undersampling jika ratio < 1.0.
    Jika ratio == 1.0 (default di exp21, exp21a, & SHINE), seluruh data dipertahankan
    tanpa merusak topologi asli jaringan.
    """
    if ratio >= 1.0:
        return df
        
    print(f"Menerapkan graph-level undersampling dengan rasio {ratio}...")
    labels = df['Is Fraud?'].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    
    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df
        
    desired_normal = int(np.floor(len(normal_idx) * ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)
        
    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)
    
    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort()
    
    return df.iloc[selected_idx].reset_index(drop=True)

def build_hetero_graph(df: pd.DataFrame, feature_cols: list, scalers_dict: dict = None) -> tuple:
    """
    Membangun graf heterogen tripartit PyTorch Geometric (HeteroData).
    Nodes:
      - transaction: Fitur transaksi bersih (feature_cols) & label target
      - user: Agregat rata-rata Amount & jumlah transaksi
      - merchant: Agregat rata-rata Amount & jumlah transaksi
    Edges:
      - (user, buys, transaction) & (transaction, rev_buys, user)
      - (merchant, sells, transaction) & (transaction, rev_sells, merchant)

    Anti-Leakage Guard:
    Jika scalers_dict is None, scaler untuk User dan Merchant di-fit pada data saat ini (hanya untuk Train).
    Jika scalers_dict disediakan, scaler dari Train digunakan untuk me-transform User dan Merchant
    pada Validation dan Test tanpa membocorkan statistik masa depan.
    """
    print(f"Membangun HeteroData graf ({len(df)} transaksi)...")
    data = HeteroData()
    
    # 1. Pemetaan ID User dan Merchant ke integer kontigu
    user_mapping = {uid: i for i, uid in enumerate(df['User'].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df['Merchant Name'].unique())}
    
    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)
    
    # 2. Agregasi fitur User dan Merchant
    user_agg = df.groupby('User').agg(
        user_avg_amount=('Amount', 'mean'),
        user_tx_count=('Amount', 'count')
    ).reset_index()
    
    merchant_agg = df.groupby('Merchant Name').agg(
        merchant_avg_amount=('Amount', 'mean'),
        merchant_tx_count=('Amount', 'count')
    ).reset_index()
    
    # Pastikan urutan sesuai dengan mapping
    user_agg['mapped_id'] = user_agg['User'].map(user_mapping)
    user_agg = user_agg.sort_values('mapped_id').reset_index(drop=True)
    
    merchant_agg['mapped_id'] = merchant_agg['Merchant Name'].map(merchant_mapping)
    merchant_agg = merchant_agg.sort_values('mapped_id').reset_index(drop=True)
    
    # Penskalaan fitur User & Merchant (Strict Anti-Leakage)
    if scalers_dict is None:
        scaler_user = StandardScaler()
        user_feats_scaled = scaler_user.fit_transform(user_agg[['user_avg_amount', 'user_tx_count']])
        
        scaler_merchant = StandardScaler()
        merchant_feats_scaled = scaler_merchant.fit_transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        
        out_scalers = {
            'scaler_user': scaler_user,
            'scaler_merchant': scaler_merchant
        }
    else:
        scaler_user = scalers_dict['scaler_user']
        scaler_merchant = scalers_dict['scaler_merchant']
        user_feats_scaled = scaler_user.transform(user_agg[['user_avg_amount', 'user_tx_count']])
        merchant_feats_scaled = scaler_merchant.transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        out_scalers = scalers_dict
        
    user_features = torch.tensor(user_feats_scaled, dtype=torch.float)
    merchant_features = torch.tensor(merchant_feats_scaled, dtype=torch.float)
    
    data['user'].x = user_features
    data['merchant'].x = merchant_features
    
    # 3. Fitur dan label Transaction
    x_tx = torch.tensor(df[feature_cols].values, dtype=torch.float)
    y_tx = torch.tensor(df['Is Fraud?'].values, dtype=torch.float)
    data['transaction'].x = x_tx
    data['transaction'].y = y_tx
    data['transaction'].time = torch.arange(num_transactions)
    
    # 4. Konstruksi Edges (Bipartit / Tripartit dua arah)
    user_src = [user_mapping[uid] for uid in df['User']]
    tx_dst = list(range(num_transactions))
    edge_index_user_tx = torch.tensor([user_src, tx_dst], dtype=torch.long)
    
    merchant_src = [merchant_mapping[mid] for mid in df['Merchant Name']]
    edge_index_merchant_tx = torch.tensor([merchant_src, tx_dst], dtype=torch.long)
    
    data['user', 'buys', 'transaction'].edge_index = edge_index_user_tx
    data['merchant', 'sells', 'transaction'].edge_index = edge_index_merchant_tx
    
    # Reverse edges untuk propagasi pesan dua arah pada GraphSAGE
    data['transaction', 'rev_buys', 'user'].edge_index = edge_index_user_tx.flip([0])
    data['transaction', 'rev_sells', 'merchant'].edge_index = edge_index_merchant_tx.flip([0])
    
    print(f"Graf selesai dibangun: {num_users} users, {num_merchants} merchants, {num_transactions} transaksi.")
    return data, out_scalers



In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroGraphSAGE(nn.Module):
    def __init__(
        self, 
        hidden_channels: int, 
        out_channels: int, 
        num_layers: int, 
        data_metadata: tuple, 
        feature_dims: dict, 
        aggr: str = 'mean',
        dropout: float = 0.2
    ):
        super().__init__()
        
        self.num_layers = num_layers
        self.aggr = aggr
        self.dropout_rate = dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        # Inisialisasi proyeksi linear untuk tiap tipe node
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(torch.randn(1, hidden_channels))
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)
                
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv(
                    in_channels=(-1, -1), # Lazy initialization
                    out_channels=hidden_channels,
                    aggr=aggr
                )
                for edge_type in data_metadata[1]
            })
            self.convs.append(conv)
            
            # BatchNorm untuk node transaction
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Classifier linear bawaan untuk monitoring val AUPRC selama pelatihan GNN
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict, return_embeddings=False):
        # 1. Proyeksi fitur awal
        h_dict = {}
        
        # Node yang memiliki fitur (transaction, user, merchant)
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
                
        # Node tanpa fitur jika ada
        for node_type, param in self.param_embeddings.items():
            num_nodes = num_nodes_dict[node_type]
            h_dict[node_type] = param.expand(num_nodes, -1)
                
        # 2. Message Passing
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            
            # ReLU, BatchNorm, dan Dropout pada transaction nodes
            for node_type in h_dict.keys():
                h_dict[node_type] = F.relu(h_dict[node_type])
                if node_type == 'transaction':
                    h_dict[node_type] = self.bns[i](h_dict[node_type])
                    h_dict[node_type] = self.dropout(h_dict[node_type])
        
        if return_embeddings:
            return h_dict['transaction']
            
        # 3. Klasifikasi biner untuk transaction nodes
        out = self.classifier(h_dict['transaction'])
        return torch.sigmoid(out).squeeze(-1)

def manual_weighted_bce(probs, targets, pos_weight, label_smoothing: float = 0.0):
    """
    Class-weighted Binary Cross-Entropy Loss untuk output probabilitas sigmoid.
    pos_weight = (num_negatives / (num_positives + 1e-7)) atau versi yang diredam (sqrt/clipped).
    """
    if label_smoothing > 0.0:
        targets_smooth = targets * (1.0 - label_smoothing) + 0.5 * label_smoothing
    else:
        targets_smooth = targets
        
    loss = - (pos_weight * targets_smooth * torch.log(probs + 1e-7) + 
              (1.0 - targets_smooth) * torch.log(1.0 - probs + 1e-7))
    return loss.mean()

def manual_unweighted_bce(probs, targets):
    """
    Unweighted Binary Cross-Entropy Loss standar untuk evaluasi metrik objektif.
    """
    loss = - (targets * torch.log(probs + 1e-7) + 
              (1.0 - targets) * torch.log(1.0 - probs + 1e-7))
    return loss.mean()



In [ ]:
%%writefile samplers.py
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

def calculate_degree_centrality(data: HeteroData, edge_type: tuple):
    """Menghitung degree centrality sederhana untuk edge type tertentu."""
    src, rel, dst = edge_type
    edge_index = data[edge_type].edge_index
    num_nodes = data[src].num_nodes
    degree = torch.zeros(num_nodes, dtype=torch.float)
    degree.scatter_add_(0, edge_index[0], torch.ones(edge_index.shape[1]))
    return degree

def apply_uniform_sampling(data: HeteroData):
    """
    Baseline: Uniform Random Sampling.
    Memberikan probabilitas yang sama (1.0) untuk semua edge.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.size(1)
        data[edge_type].edge_weight = torch.ones(num_edges, dtype=torch.float)
    return data

def apply_topology_aware_sampling(data: HeteroData):
    """
    Topology-Aware Sampling (Inverse Log-Degree Centrality - v5 Stabilized).
    Menggunakan inverse sublinear log1p(degree) untuk memprioritaskan edge struktural
    tanpa disparitas bobot yang ekstrem.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Sublinear scaling: log1p(degree)
        log_deg = torch.log1p(deg_src[src_nodes])
        weights = 1.0 / (log_deg + 1e-5)
        min_val = weights.min()
        max_val = weights.max()
        if (max_val - min_val) > 1e-5:
            weights = (weights - min_val) / (max_val - min_val)
        else:
            weights = torch.zeros_like(weights)
        data[edge_type].edge_weight = weights + 0.1
    return data

def apply_importance_based_sampling(data: HeteroData):
    """
    Importance-Based Sampling (Log-Degree Centrality Proxy - v5 Stabilized).
    Menggunakan transformasi sublinear log1p untuk meredam varians sampling
    akibat super-hub merchant dan mencegah anomali ledakan False Positive di akhir training.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Sublinear scaling: log1p(degree) meredam lonjakan super-hubs
        log_deg = torch.log1p(deg_src[src_nodes])
        min_val = log_deg.min()
        max_val = log_deg.max()
        if (max_val - min_val) > 1e-5:
            importance_weights = (log_deg - min_val) / (max_val - min_val)
        else:
            importance_weights = torch.zeros_like(log_deg)
        data[edge_type].edge_weight = importance_weights + 0.1
    return data

def get_neighbor_loader(
    data: HeteroData, 
    batch_size: int = 1024, 
    num_neighbors: list = [15, 10], 
    method: str = 'uniform', 
    shuffle: bool = True, 
    input_nodes=None, 
    num_workers: int = 0
):
    """
    Mengembalikan PyG NeighborLoader berdasarkan strategi neighbor sampling yang dipilih.
    """
    if method == 'uniform':
        data = apply_uniform_sampling(data)
    elif method == 'topology':
        data = apply_topology_aware_sampling(data)
    elif method == 'importance':
        data = apply_importance_based_sampling(data)
    else:
        raise ValueError(f"Metode sampling tidak dikenal: {method}")

    if input_nodes is None:
        input_nodes = ('transaction', torch.arange(data['transaction'].num_nodes))

    loader = NeighborLoader(
        data,
        num_neighbors=num_neighbors,
        batch_size=batch_size,
        input_nodes=input_nodes,
        shuffle=shuffle,
        weight_attr='edge_weight',
        num_workers=num_workers
    )
    return loader



In [ ]:
%%writefile visualize_results.py
import os
import json
import argparse
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from sklearn.metrics import precision_recall_curve, roc_curve, confusion_matrix, ConfusionMatrixDisplay

def setup_plot_style():
    plt.rcParams.update({
        'font.size': 11,
        'axes.labelsize': 12,
        'axes.titlesize': 13,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight'
    })

def plot_feature_importance(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Feature Importance XGBoost: Membedakan secara visual kontribusi
    fitur tabular bersih vs representasi graf 128-d GNN.
    """
    for method in methods:
        imp_path = result_dir / f"exp21a_{method}_feature_importance.json"
        if not imp_path.exists():
            continue
            
        with open(imp_path, 'r') as f:
            data = json.load(f)
            
        top_features = data.get('top_features', [])[:15]
        if not top_features:
            continue
            
        names = [item['feature_name'] for item in reversed(top_features)]
        gains = [item['gain'] for item in reversed(top_features)]
        colors = ['#1f77b4' if item['is_gnn'] else '#ff7f0e' for item in reversed(top_features)]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        bars = ax.barh(names, gains, color=colors, edgecolor='black', linewidth=0.7)
        
        ax.set_xlabel('Split Gain (Tingkat Kepentingan Keputusan)')
        ax.set_title(f'EXP21A: Feature Importance XGBoost ({method.upper()})\n'
                     f'Tabular: {data["pct_tabular_gain"]:.1f}% | GNN Embeddings: {data["pct_gnn_gain"]:.1f}%', fontweight='bold')
        ax.grid(axis='x', linestyle='--', alpha=0.6)
        
        # Legend manual
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='#ff7f0e', edgecolor='black', label='Clean Tabular Feature'),
            Patch(facecolor='#1f77b4', edgecolor='black', label='GraphSAGE Representation')
        ]
        ax.legend(handles=legend_elements, loc='lower right')
        
        plt.tight_layout()
        save_path = output_dir / f"exp21a_{method}_feature_importance.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def plot_pr_and_roc(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Kurva Precision-Recall dan ROC pada Test Set masa depan.
    """
    fig, (ax_pr, ax_roc) = plt.subplots(1, 2, figsize=(14, 6))
    colors = {'uniform': '#1f77b4', 'topology': '#2ca02c', 'importance': '#d62728'}
    
    found_any = False
    for method in methods:
        pred_path = result_dir / f"exp21a_{method}_predictions.npz"
        res_path = result_dir / f"exp21a_{method}_results.json"
        if not (pred_path.exists() and res_path.exists()):
            continue
            
        found_any = True
        npz = np.load(pred_path)
        test_probs = npz['test_probs']
        test_labels = npz['test_labels']
        
        with open(res_path, 'r') as f:
            res = json.load(f)
            
        auprc = res['metrics_threshold_independent']['auprc']
        roc_auc = res['metrics_threshold_independent']['roc_auc']
        tau = res['metrics_optimal_threshold']['threshold']
        f1 = res['metrics_optimal_threshold']['f1_score']
        
        # Kurva PR
        prec, rec, _ = precision_recall_curve(test_labels, test_probs)
        ax_pr.plot(rec, prec, label=f"{method.capitalize()} (AUCPR = {auprc:.4f})", color=colors.get(method, 'blue'), lw=2)
        
        # Kurva ROC
        fpr, tpr, _ = roc_curve(test_labels, test_probs)
        ax_roc.plot(fpr, tpr, label=f"{method.capitalize()} (AUC = {roc_auc:.4f})", color=colors.get(method, 'blue'), lw=2)

    if found_any:
        # Baseline PR
        base_rate = float(test_labels.mean())
        ax_pr.axhline(base_rate, color='grey', linestyle='--', label=f'Random Baseline ({base_rate*100:.2f}%)')
        ax_pr.set_xlabel('Recall')
        ax_pr.set_ylabel('Precision')
        ax_pr.set_title('Precision-Recall Curves (Test Set 60 Hari)', fontweight='bold')
        ax_pr.legend(loc='upper right')
        ax_pr.grid(True, linestyle='--', alpha=0.5)

        # Baseline ROC
        ax_roc.plot([0, 1], [0, 1], color='grey', linestyle='--', label='Random Chance (AUC = 0.50)')
        ax_roc.set_xlabel('False Positive Rate')
        ax_roc.set_ylabel('True Positive Rate (Recall)')
        ax_roc.set_title('ROC Curves (Test Set 60 Hari)', fontweight='bold')
        ax_roc.legend(loc='lower right')
        ax_roc.grid(True, linestyle='--', alpha=0.5)

        plt.tight_layout()
        save_path = output_dir / "exp21a_pr_roc_curves.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def plot_learning_curves(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Kurva Pelatihan Loss dan Val AUPRC per Epoch.
    """
    for method in methods:
        hist_path = result_dir / f"exp21a_{method}_history.json"
        if not hist_path.exists():
            continue
            
        with open(hist_path, 'r') as f:
            hist = json.load(f)
            
        epochs = hist['epoch']
        train_loss = hist['train_loss']
        val_auprc = hist['val_auprc']
        val_f1 = hist['val_f1']
        
        fig, ax1 = plt.subplots(figsize=(8, 5))
        color = 'tab:red'
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Train Weighted Loss', color=color)
        ax1.plot(epochs, train_loss, color=color, lw=2, marker='o', label='Train Loss')
        ax1.tick_params(axis='y', labelcolor=color)
        ax1.grid(True, linestyle='--', alpha=0.5)

        ax2 = ax1.twinx()
        color = 'tab:blue'
        ax2.set_ylabel('Val AUPRC / F1', color=color)
        ax2.plot(epochs, val_auprc, color=color, lw=2, marker='s', label='Val AUPRC')
        ax2.plot(epochs, val_f1, color='tab:green', lw=1.5, linestyle='--', marker='^', label='Val F1')
        ax2.tick_params(axis='y', labelcolor=color)

        plt.title(f'EXP21A: GNN Training Learning Curves ({method.capitalize()})', fontweight='bold')
        fig.tight_layout()
        save_path = output_dir / f"exp21a_{method}_learning_curve.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def main():
    parser = argparse.ArgumentParser(description="Generasi Visualisasi Hasil Eksperimen 21A")
    parser.add_argument("--result-dir", type=str, default="../../result/exp21a", help="Direktori hasil JSON/NPZ")
    parser.add_argument("--output-dir", type=str, default="../../result/exp21a/plots", help="Direktori penyimpanan plot")
    args = parser.parse_args()

    result_dir = Path(args.result_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    setup_plot_style()
    methods = ["uniform", "topology", "importance"]

    print("="*70)
    print("  MEMULAI GENERASI VISUALISASI ILMIAH EXP21A")
    print("="*70)
    plot_feature_importance(result_dir, output_dir, methods)
    plot_pr_and_roc(result_dir, output_dir, methods)
    plot_learning_curves(result_dir, output_dir, methods)
    print("="*70)
    print(f"Seluruh visualisasi berhasil disimpan di: {output_dir}")
    print("="*70)

if __name__ == "__main__":
    main()



In [ ]:
%%writefile main.py
import argparse
import time
import json
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from sklearn.metrics import (
    f1_score, recall_score, precision_score, 
    accuracy_score, roc_auc_score, average_precision_score, 
    confusion_matrix
)
from xgboost import XGBClassifier

from config import load_config
from data_preprocessing import load_and_split_raw_data, apply_strict_anti_leakage_preprocessing
from graph_builder import build_hetero_graph, graph_level_undersample
from samplers import get_neighbor_loader
from model import HeteroGraphSAGE, manual_weighted_bce, manual_unweighted_bce

def extract_features(model, loader, device, total_samples, orig_feat_dim, emb_dim):
    """
    Mengekstrak embedding GNN dan menggabungkannya dengan fitur tabular transaksi yang bersih.
    Menggunakan pre-alokasi memori numpy untuk efisiensi RAM maksimal.
    """
    model.eval()
    final_features = np.empty((total_samples, orig_feat_dim + emb_dim), dtype=np.float32)
    all_targets = np.empty((total_samples,), dtype=np.int64)
    
    start_time = time.time()
    current_idx = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            
            embeddings = model(
                batch.x_dict, 
                batch.edge_index_dict, 
                num_nodes_dict=num_nodes_dict, 
                return_embeddings=True
            )
            
            batch_size = batch['transaction'].batch_size
            next_idx = min(current_idx + batch_size, total_samples)
            actual_count = next_idx - current_idx
            
            final_features[current_idx:next_idx, :orig_feat_dim] = batch['transaction'].x[:actual_count].cpu().numpy()
            final_features[current_idx:next_idx, orig_feat_dim:] = embeddings[:actual_count].cpu().numpy()
            all_targets[current_idx:next_idx] = batch['transaction'].y[:actual_count].cpu().numpy()
            
            current_idx = next_idx
            
    inference_time = time.time() - start_time
    
    if current_idx < total_samples:
        final_features = final_features[:current_idx]
        all_targets = all_targets[:current_idx]
        
    return final_features, all_targets, inference_time

def find_optimal_threshold(val_probs, val_labels, thresholds=None, min_precision=0.10):
    """
    Mencari threshold optimal tau* yang memaksimalkan F1-score pada Validation set,
    dengan konstrain presisi minimum yang masuk akal secara operasional perbankan.
    """
    if thresholds is None:
        thresholds = np.arange(0.02, 0.70, 0.005)
        
    best_tau = 0.5
    best_f1 = 0.0
    best_metrics = {}
    
    for tau in thresholds:
        preds = (val_probs >= tau).astype(int)
        f1 = f1_score(val_labels, preds, zero_division=0)
        prec = precision_score(val_labels, preds, zero_division=0)
        rec = recall_score(val_labels, preds, zero_division=0)
        
        # Prioritaskan F1 tertinggi
        if f1 > best_f1:
            best_f1 = f1
            best_tau = float(tau)
            best_metrics = {
                "tau": float(tau),
                "val_f1": float(f1),
                "val_precision": float(prec),
                "val_recall": float(rec)
            }
            
    if best_f1 == 0.0:
        best_tau = 0.5
        best_metrics = {
            "tau": 0.5,
            "val_f1": 0.0,
            "val_precision": 0.0,
            "val_recall": 0.0
        }
        
    return best_tau, best_metrics

def evaluate_gnn(model, loader, device, pos_weight):
    """
    Mengevaluasi representasi GNN pada validation loader.
    """
    model.eval()
    total_loss = 0.0
    total_loss_unw = 0.0
    total_samples = 0
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            batch_size_current = batch['transaction'].batch_size
            probs = probs[:batch_size_current]
            targets = batch['transaction'].y[:batch_size_current]
            loss = manual_weighted_bce(probs, targets, pos_weight)
            loss_unw = manual_unweighted_bce(probs, targets)
            
            total_loss += loss.item() * batch_size_current
            total_loss_unw += loss_unw.item() * batch_size_current
            total_samples += batch_size_current
            all_probs.append(probs.cpu())
            all_targets.append(targets.cpu())
            
    all_probs = torch.cat(all_probs).numpy().ravel()
    all_targets = torch.cat(all_targets).numpy().ravel()
    avg_loss = total_loss / max(1, total_samples)
    avg_loss_unw = total_loss_unw / max(1, total_samples)
    
    aucpr = float(average_precision_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.0
    roc_auc = float(roc_auc_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.0
    
    best_tau, tau_stats = find_optimal_threshold(all_probs, all_targets)
    preds_tau = (all_probs >= best_tau).astype(int)
    f1_tau = float(f1_score(all_targets, preds_tau, zero_division=0))
    rec_tau = float(recall_score(all_targets, preds_tau, zero_division=0))
    prec_tau = float(precision_score(all_targets, preds_tau, zero_division=0))
    
    return {
        'loss': float(avg_loss),
        'loss_unweighted': float(avg_loss_unw),
        'auprc': aucpr,
        'roc_auc': roc_auc,
        'best_tau': best_tau,
        'f1': f1_tau,
        'recall': rec_tau,
        'precision': prec_tau
    }

def main():
    parser = argparse.ArgumentParser(description="EXP21A: Anti-Leakage Data Splitting & Attribute Clean Pipeline")
    parser.add_argument("--config", type=str, default="config.reference.yaml", help="Path konfigurasi YAML")
    parser.add_argument("--sampling", type=str, default=None, choices=["uniform", "topology", "importance"])
    parser.add_argument("--nrows", type=int, default=None, help="Batas baris untuk smoke testing")
    parser.add_argument("--reference-date", type=str, default=None, help="Override tanggal T0 (default dari config)")
    parser.add_argument("--feature-mode", type=str, default=None, choices=["clean", "clean_basic", "legacy_leaky"])
    args = parser.parse_args()

    cfg = load_config(args.config)
    sampling_method = args.sampling if args.sampling else cfg['sampling']['method']
    nrows = args.nrows if args.nrows is not None else cfg['data'].get('max_rows', None)
    
    time_window_cfg = cfg.get('time_window', {})
    reference_date = args.reference_date if args.reference_date else time_window_cfg.get('reference_date', '2018-01-01')
    train_days = time_window_cfg.get('train_days', 300)
    val_days = time_window_cfg.get('val_days', 30)
    gap_days = time_window_cfg.get('gap_days', 8)
    test_days = time_window_cfg.get('test_days', 60)
    
    feature_mode = args.feature_mode if args.feature_mode else cfg.get('features', {}).get('mode', 'clean')
    use_cyclical_time = cfg.get('features', {}).get('use_cyclical_time', True)

    print("="*80)
    print(f"  EKSPERIMEN 21A: AUDIT & REMEDIASI KEBOCORAN ATRIBUT + STRICT SPLITTING")
    print(f"  Sampling Method   : {sampling_method.upper()}")
    print(f"  Feature Mode      : {feature_mode.upper()} (Zip & Merchant City Excluded: {feature_mode != 'legacy_leaky'})")
    print(f"  Reference Date T0 : {reference_date} (Train: {train_days}d, Val: {val_days}d, Gap: {gap_days}d, Test: {test_days}d)")
    print("="*80)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device yang digunakan: {device}")

    model_dir = Path(cfg.get('output', {}).get('model_dir', '../../model/exp21a'))
    result_dir = Path(cfg.get('output', {}).get('result_dir', '../../result/exp21a'))
    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load Raw Data & Split Sebelum Preprocessing (Strict Protocol)
    dataset_path = cfg['data']['transactions']
    start_year = cfg['data'].get('start_year', 2016)
    
    prep_start = time.time()
    train_df, val_df, test_df, gap_df, window_stats = load_and_split_raw_data(
        file_path=dataset_path,
        start_year=start_year,
        nrows=nrows,
        reference_date=reference_date,
        train_days=train_days,
        val_days=val_days,
        gap_days=gap_days,
        test_days=test_days
    )
    
    # 2. Strict Anti-Leakage Preprocessing
    train_df, val_df, test_df, gap_df, feature_cols, prep_artifacts = apply_strict_anti_leakage_preprocessing(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        gap_df=gap_df,
        feature_mode=feature_mode,
        use_cyclical_time=use_cyclical_time
    )
    data_prep_time = time.time() - prep_start
    print(f"Data preprocessing selesai dalam {data_prep_time:.2f} detik.")

    # Simpan statistik jendela waktu
    stats_path = result_dir / "window_statistics.json"
    with open(stats_path, 'w') as f:
        json.dump(window_stats, f, indent=4)
    print(f"Statistik jendela waktu tersimpan ke: {stats_path}")

    # 3. Undersampling Graf (Default: 1.0 = Tanpa Undersampling)
    undersample_ratio = cfg['sampling'].get('undersampling_ratio', 1.0)
    train_df_sampled = graph_level_undersample(train_df, ratio=undersample_ratio)

    # 4. Konstruksi Graf Heterogen (PyG)
    print("\n--- [Langkah 3: Membangun Graf Train (PyG)] ---")
    train_data, entity_scalers = build_hetero_graph(train_df_sampled, feature_cols, scalers_dict=None)
    
    batch_size = cfg['training'].get('batch_size', 1024)
    num_neighbors = cfg['training'].get('num_neighbors', [15, 10])
    
    train_loader = get_neighbor_loader(
        train_data, 
        batch_size=batch_size, 
        num_neighbors=num_neighbors, 
        method=sampling_method, 
        shuffle=True
    )

    # Siapkan Graf Validation (Train + Val)
    print("Membangun Graf Validation (Train + Val) dengan entity scaler Train...")
    val_graph_df = pd.concat([train_df, val_df], ignore_index=True)
    val_data, _ = build_hetero_graph(val_graph_df, feature_cols, scalers_dict=entity_scalers)
    val_node_indices = torch.arange(len(train_df), len(val_graph_df))
    val_loader = get_neighbor_loader(
        val_data,
        batch_size=4096,
        num_neighbors=num_neighbors,
        method=sampling_method,
        shuffle=False,
        input_nodes=('transaction', val_node_indices)
    )

    # 5. Definisi Arsitektur Model GNN (HeteroGraphSAGE)
    feature_dims = {
        'transaction': len(feature_cols),
        'user': 2,
        'merchant': 2
    }
    hidden_channels = cfg['model'].get('hidden_channels', 128)
    out_channels = cfg['model'].get('out_channels', 1)
    num_layers = cfg['model'].get('num_layers', 2)
    aggr = cfg['model'].get('aggr', 'mean')
    dropout = float(cfg['model'].get('dropout', 0.2))

    model = HeteroGraphSAGE(
        hidden_channels=hidden_channels,
        out_channels=out_channels,
        num_layers=num_layers,
        data_metadata=train_data.metadata(),
        feature_dims=feature_dims,
        aggr=aggr,
        dropout=dropout
    ).to(device)

    # Hitung bobot penalti kelas
    num_pos = int(train_data['transaction'].y.sum().item())
    num_neg = int(train_data['transaction'].num_nodes - num_pos)
    raw_pos_weight = float(num_neg / (num_pos + 1e-7))

    loss_weight_strategy = cfg.get('training', {}).get('loss_weight_strategy', 'sqrt')
    if loss_weight_strategy == 'sqrt':
        eff_pos_weight = float(np.sqrt(raw_pos_weight))
    elif loss_weight_strategy == 'clipped':
        max_pw = float(cfg.get('training', {}).get('max_pos_weight', 50.0))
        eff_pos_weight = float(min(raw_pos_weight, max_pw))
    else:
        eff_pos_weight = float(raw_pos_weight)

    pos_weight = torch.tensor([eff_pos_weight]).to(device)
    print(f"GNN Class-Weight pos_weight ({loss_weight_strategy}): {pos_weight.item():.2f} (raw ratio: {raw_pos_weight:.2f})")

    lr = float(cfg['training'].get('learning_rate', 0.001))
    weight_decay = float(cfg['training'].get('weight_decay', 1e-4))
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Learning Rate Scheduler
    use_scheduler = cfg.get('training', {}).get('use_lr_scheduler', True)
    if use_scheduler:
        scheduler_factor = float(cfg['training'].get('lr_scheduler_factor', 0.5))
        scheduler_patience = int(cfg['training'].get('lr_scheduler_patience', 3))
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=scheduler_factor, patience=scheduler_patience, min_lr=1e-5
        )
    else:
        scheduler = None

    # 6. Phase 1: Pelatihan GraphSAGE Encoder
    print("\n--- [Phase 1: Pelatihan HeteroGraphSAGE Encoder] ---")
    epochs = cfg['training'].get('epochs', 20)
    early_stopping_patience = int(cfg['training'].get('early_stopping_patience', 7))
    grad_clip_norm = float(cfg['training'].get('grad_clip_norm', 5.0))

    gnn_epoch_history = {
        'epoch': [], 'train_loss': [], 'train_auprc': [], 'train_f1': [],
        'val_loss': [], 'val_auprc': [], 'val_f1': [], 'val_recall': [], 'val_precision': [], 'lr': []
    }

    best_val_auprc = -1.0
    best_epoch = 1
    best_model_state = None
    patience_counter = 0
    best_model_path = model_dir / f"exp21a_{sampling_method}_best_gnn.pt"

    gnn_train_start = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_samples = 0
        epoch_probs = []
        epoch_targets = []
        
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            
            batch_size_current = batch['transaction'].batch_size
            probs = probs[:batch_size_current]
            targets = batch['transaction'].y[:batch_size_current]
            
            loss = manual_weighted_bce(probs, targets, pos_weight)
            loss.backward()
            
            if grad_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
                
            optimizer.step()
            
            total_loss += loss.item() * batch_size_current
            total_samples += batch_size_current
            epoch_probs.append(probs.detach().cpu())
            epoch_targets.append(targets.cpu())
            
        train_probs_cat = torch.cat(epoch_probs).numpy().ravel()
        train_targets_cat = torch.cat(epoch_targets).numpy().ravel()
        train_loss = total_loss / max(1, total_samples)
        train_auprc = float(average_precision_score(train_targets_cat, train_probs_cat)) if len(np.unique(train_targets_cat)) > 1 else 0.0
        train_f1 = float(f1_score(train_targets_cat, (train_probs_cat >= 0.5).astype(int), zero_division=0))
        
        # Evaluasi validation set di akhir epoch
        val_metrics = evaluate_gnn(model, val_loader, device, pos_weight)
        current_lr = optimizer.param_groups[0]['lr']
        
        if val_metrics['auprc'] > best_val_auprc:
            best_val_auprc = val_metrics['auprc']
            best_epoch = epoch + 1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(model.state_dict(), best_model_path)
            patience_counter = 0
            mark = "★ BEST"
        else:
            patience_counter += 1
            mark = ""
            
        if scheduler is not None:
            scheduler.step(val_metrics['auprc'])
            
        gnn_epoch_history['epoch'].append(epoch + 1)
        gnn_epoch_history['train_loss'].append(train_loss)
        gnn_epoch_history['train_auprc'].append(train_auprc)
        gnn_epoch_history['train_f1'].append(train_f1)
        gnn_epoch_history['val_loss'].append(val_metrics['loss'])
        gnn_epoch_history['val_auprc'].append(val_metrics['auprc'])
        gnn_epoch_history['val_f1'].append(val_metrics['f1'])
        gnn_epoch_history['val_recall'].append(val_metrics['recall'])
        gnn_epoch_history['val_precision'].append(val_metrics['precision'])
        gnn_epoch_history['lr'].append(current_lr)
        
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] "
              f"Train Loss: {train_loss:.4f} | Train AUPRC: {train_auprc:.4f} || "
              f"Val AUPRC: {val_metrics['auprc']:.4f} | Val F1: {val_metrics['f1']:.4f} | "
              f"Val Rec: {val_metrics['recall']*100:.1f}% | Val Prec: {val_metrics['precision']*100:.1f}% | "
              f"Patience: {patience_counter}/{early_stopping_patience} {mark}")
              
        if patience_counter >= early_stopping_patience:
            print(f"\n[EARLY STOPPING] Pelatihan dihentikan lebih awal pada Epoch {epoch+1}. Best Epoch: {best_epoch} (Val AUPRC: {best_val_auprc:.4f})")
            break

    gnn_train_time = time.time() - gnn_train_start
    print(f"\nPelatihan GraphSAGE selesai dalam {gnn_train_time:.2f} detik. Memulihkan bobot terbaik dari Epoch {best_epoch}...")
    if best_model_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})

    del train_loader, optimizer
    gc.collect()
    torch.cuda.empty_cache()

    # 7. Phase 2: Ekstraksi Fitur (Train, Validation, Test)
    print("\n--- [Phase 2: Ekstraksi Representasi Induktif (Embeddings + Clean Tabular)] ---")
    orig_dim = len(feature_cols)
    emb_dim = hidden_channels
    
    # Ekstraksi Train
    print(f"Mengekstrak fitur Train ({len(train_df_sampled)} sampel)...")
    train_ext_loader = get_neighbor_loader(
        train_data, batch_size=4096, num_neighbors=num_neighbors, method=sampling_method, shuffle=False
    )
    train_features, train_labels, inf_time_train = extract_features(
        model, train_ext_loader, device, total_samples=len(train_df_sampled), orig_feat_dim=orig_dim, emb_dim=emb_dim
    )
    del train_data, train_ext_loader
    gc.collect()
    torch.cuda.empty_cache()

    # Ekstraksi Validation
    print(f"Mengekstrak fitur Validation ({len(val_node_indices)} sampel)...")
    val_features, val_labels, inf_time_val_gnn = extract_features(
        model, val_loader, device, total_samples=len(val_node_indices), orig_feat_dim=orig_dim, emb_dim=emb_dim
    )
    del val_data, val_loader, val_graph_df
    gc.collect()
    torch.cuda.empty_cache()

    # Ekstraksi Test (Train + Val + Test)
    print(f"Mengekstrak fitur Test ({len(test_df)} sampel)...")
    test_graph_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    test_data, _ = build_hetero_graph(test_graph_df, feature_cols, scalers_dict=entity_scalers)
    test_node_indices = torch.arange(len(train_df) + len(val_df), len(test_graph_df))
    test_loader = get_neighbor_loader(
        test_data, batch_size=4096, num_neighbors=num_neighbors, method=sampling_method, shuffle=False,
        input_nodes=('transaction', test_node_indices)
    )
    test_features, test_labels, inf_time_test_gnn = extract_features(
        model, test_loader, device, total_samples=len(test_node_indices), orig_feat_dim=orig_dim, emb_dim=emb_dim
    )
    del test_data, test_loader, test_graph_df, model
    gc.collect()
    torch.cuda.empty_cache()

    # 8. Phase 3: Klasifikasi dengan XGBoost Ter-regularisasi
    print("\n--- [Phase 3: Pelatihan & Prediksi XGBoost Classifier Ter-regularisasi] ---")
    cls_cfg = cfg.get('classifier', {})
    num_neg_xgb = int((train_labels == 0).sum())
    num_pos_xgb = int((train_labels == 1).sum())
    raw_xgb_ratio = float(num_neg_xgb / num_pos_xgb) if num_pos_xgb > 0 else 1.0

    scale_pos_strategy = cls_cfg.get('scale_pos_weight_strategy', 'sqrt')
    if scale_pos_strategy == 'sqrt':
        scale_pos_weight = float(np.sqrt(raw_xgb_ratio))
    elif scale_pos_strategy == 'moderate':
        scale_pos_weight = 10.0
    elif scale_pos_strategy == 'none':
        scale_pos_weight = 1.0
    else: # full
        scale_pos_weight = raw_xgb_ratio

    print(f"XGBoost scale_pos_weight ({scale_pos_strategy}): {scale_pos_weight:.2f} (raw ratio: {raw_xgb_ratio:.2f})")
    
    # Feature names untuk audit importance
    feature_names = list(feature_cols) + [f"gnn_emb_{i}" for i in range(emb_dim)]

    xgb_params = {
        'n_estimators': cls_cfg.get('n_estimators', 100),
        'max_depth': cls_cfg.get('max_depth', 4),
        'learning_rate': cls_cfg.get('learning_rate', 0.05),
        'min_child_weight': cls_cfg.get('min_child_weight', 10),
        'subsample': cls_cfg.get('subsample', 0.8),
        'colsample_bytree': cls_cfg.get('colsample_bytree', 0.8),
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'eval_metric': ['logloss', 'aucpr', 'auc']
    }

    eval_set = [(train_features, train_labels), (val_features, val_labels)]
    xgb_start = time.time()
    try:
        xgb_model = XGBClassifier(**xgb_params, tree_method='hist', device='cuda')
        xgb_model.fit(train_features, train_labels, eval_set=eval_set, verbose=False)
    except Exception as e:
        print(f"XGBoost GPU fit gagal ({e}), beralih ke CPU...")
        xgb_model = XGBClassifier(**xgb_params, tree_method='hist', n_jobs=-1)
        xgb_model.fit(train_features, train_labels, eval_set=eval_set, verbose=False)
    xgb_train_time = time.time() - xgb_start
    print(f"XGBoost training selesai dalam {xgb_train_time:.2f} detik.")

    # Simpan model XGBoost
    xgb_model_path = model_dir / f"exp21a_{sampling_method}_xgb.json"
    xgb_model.save_model(xgb_model_path)
    print(f"Model XGBoost tersimpan di: {xgb_model_path}")

    # AUDIT FEATURE IMPORTANCE XGBOOST (Tabular vs GNN)
    booster = xgb_model.get_booster()
    # Petakan f0, f1... ke nama kolom
    score_gain = booster.get_score(importance_type='gain')
    score_weight = booster.get_score(importance_type='weight')
    
    importance_list = []
    total_gain_all = 0.0
    total_gain_tabular = 0.0
    total_gain_gnn = 0.0

    for idx, name in enumerate(feature_names):
        f_key = f"f{idx}"
        gain = score_gain.get(f_key, 0.0)
        weight = score_weight.get(f_key, 0)
        is_gnn = name.startswith("gnn_emb_")
        total_gain_all += gain
        if is_gnn:
            total_gain_gnn += gain
        else:
            total_gain_tabular += gain
            
        importance_list.append({
            "feature_index": idx,
            "feature_name": name,
            "is_gnn": is_gnn,
            "gain": float(gain),
            "weight": int(weight)
        })

    importance_list.sort(key=lambda x: x['gain'], reverse=True)
    pct_tabular = (total_gain_tabular / max(1e-5, total_gain_all)) * 100
    pct_gnn = (total_gain_gnn / max(1e-5, total_gain_all)) * 100

    print("\n" + "="*80)
    print("  AUDIT FEATURE IMPORTANCE XGBOOST (TABULAR VS GRAPH REPRESENTATION)")
    print("="*80)
    print(f"Total Gain Tabular Fitur : {total_gain_tabular:12.2f} ({pct_tabular:5.2f}%)")
    print(f"Total Gain GNN Fitur     : {total_gain_gnn:12.2f} ({pct_gnn:5.2f}%)")
    print("-" * 80)
    print("Top 15 Fitur Paling Berpengaruh Berdasarkan Gain:")
    for rank, item in enumerate(importance_list[:15], 1):
        tipe = "[GNN]" if item['is_gnn'] else "[TAB]"
        print(f"  {rank:2d}. {tipe} {item['feature_name']:20s} | Gain: {item['gain']:10.2f} | Split Count: {item['weight']:4d}")
    print("="*80)

    # Simpan hasil audit feature importance
    importance_summary = {
        "total_gain_all": float(total_gain_all),
        "total_gain_tabular": float(total_gain_tabular),
        "total_gain_gnn": float(total_gain_gnn),
        "pct_tabular_gain": float(pct_tabular),
        "pct_gnn_gain": float(pct_gnn),
        "top_features": importance_list[:30]
    }
    with open(result_dir / f"exp21a_{sampling_method}_feature_importance.json", 'w') as f:
        json.dump(importance_summary, f, indent=4)

    # 9. Phase 4: Kalibrasi Threshold Dinamis pada Validation & Evaluasi Test Set
    print("\n--- [Phase 4: Kalibrasi Threshold Dinamis & Evaluasi Final Test Set] ---")
    val_probs = xgb_model.predict_proba(val_features)[:, 1]
    best_tau, val_tuning_stats = find_optimal_threshold(val_probs, val_labels)
    print(f"Optimal Threshold (tau*) dari Validation: {best_tau:.3f} (Val F1: {val_tuning_stats['val_f1']:.4f}, Val Prec: {val_tuning_stats['val_precision']:.4f}, Val Rec: {val_tuning_stats['val_recall']:.4f})")

    test_pred_start = time.time()
    test_probs = xgb_model.predict_proba(test_features)[:, 1]
    inf_time_xgb = time.time() - test_pred_start
    total_test_inf_time = inf_time_test_gnn + inf_time_xgb

    # Evaluasi @ 0.5 (Default baseline)
    preds_05 = (test_probs >= 0.5).astype(int)
    f1_05 = float(f1_score(test_labels, preds_05, zero_division=0))
    recall_05 = float(recall_score(test_labels, preds_05, zero_division=0))
    precision_05 = float(precision_score(test_labels, preds_05, zero_division=0))

    # Evaluasi @ tau* (Optimal)
    preds_tau = (test_probs >= best_tau).astype(int)
    f1_tau = float(f1_score(test_labels, preds_tau, zero_division=0))
    f1_macro_tau = float(f1_score(test_labels, preds_tau, average='macro', zero_division=0))
    recall_tau = float(recall_score(test_labels, preds_tau, zero_division=0))
    precision_tau = float(precision_score(test_labels, preds_tau, zero_division=0))
    accuracy_tau = float(accuracy_score(test_labels, preds_tau))

    cm = confusion_matrix(test_labels, preds_tau)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = [int(v) for v in cm.ravel()]
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        gmean = float(np.sqrt(recall_tau * specificity))
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
        specificity, gmean = 0.0, 0.0

    roc_auc = float(roc_auc_score(test_labels, test_probs)) if len(np.unique(test_labels)) > 1 else 0.0
    auprc = float(average_precision_score(test_labels, test_probs)) if len(np.unique(test_labels)) > 1 else 0.0

    # Metrik Operasional Perbankan: Lift@1% dan TP@500
    top_1pct_count = max(1, int(round(len(test_labels) * 0.01)))
    top_1pct_idx = np.argsort(test_probs)[::-1][:top_1pct_count]
    precision_1pct = float(test_labels[top_1pct_idx].mean())
    base_fraud_rate = float(test_labels.mean())
    lift_1pct = float(precision_1pct / base_fraud_rate) if base_fraud_rate > 0 else 0.0

    top_500_idx = np.argsort(test_probs)[::-1][:500]
    tp_at_500 = int(test_labels[top_500_idx].sum())
    precision_at_500 = float(tp_at_500 / 500)

    print("\n" + "="*80)
    print(f"  HASIL EVALUASI AKHIR PADA TEST SET MASA DEPAN (60 HARI, N={len(test_labels):,})")
    print("="*80)
    print(f"  • Metrik Bebas Threshold (Threshold-Independent):")
    print(f"      - AUCPR (Area Under PR Curve) : {auprc:.4f}")
    print(f"      - ROC-AUC                     : {roc_auc:.4f}")
    print(f"  • Metrik Operasional Perbankan:")
    print(f"      - Lift @ 1% Transaksi Teratas : {lift_1pct:.2f}x")
    print(f"      - TP @ 500 Alert Harian       : {tp_at_500} / 500 ({precision_at_500*100:.1f}%)")
    print(f"  • Metrik pada Threshold Optimal tau* = {best_tau:.3f}:")
    print(f"      - F1-Score                    : {f1_tau:.4f}")
    print(f"      - Recall                      : {recall_tau*100:.2f}% (TP={tp}, FN={fn})")
    print(f"      - Precision                   : {precision_tau*100:.2f}% (FP={fp})")
    print(f"      - Akurasi                     : {accuracy_tau*100:.4f}%")
    print(f"      - G-Mean                      : {gmean:.4f}")
    print(f"  • Metrik pada Baseline Threshold 0.50:")
    print(f"      - F1-Score: {f1_05:.4f} | Recall: {recall_05*100:.2f}% | Precision: {precision_05*100:.2f}%")
    print("="*80)

    # Simpan hasil akhir ke JSON
    results = {
        "experiment_id": "EXP21A",
        "experiment_name": "Strict Anti-Leakage & Attribute Clean Pipeline",
        "sampling_method": sampling_method,
        "feature_mode": feature_mode,
        "feature_cols": feature_cols,
        "num_features_tabular": orig_dim,
        "num_features_gnn": emb_dim,
        "time_window": {
            "reference_date": reference_date,
            "train_days": train_days,
            "val_days": val_days,
            "gap_days": gap_days,
            "test_days": test_days
        },
        "threshold_calibration": {
            "optimal_tau": float(best_tau),
            "val_f1": float(val_tuning_stats.get('val_f1', 0.0)),
            "val_precision": float(val_tuning_stats.get('val_precision', 0.0)),
            "val_recall": float(val_tuning_stats.get('val_recall', 0.0))
        },
        "metrics_optimal_threshold": {
            "threshold": float(best_tau),
            "f1_score": f1_tau,
            "f1_macro": f1_macro_tau,
            "recall": recall_tau,
            "precision": precision_tau,
            "accuracy": accuracy_tau,
            "gmean": gmean,
            "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp}
        },
        "metrics_default_threshold_05": {
            "threshold": 0.5,
            "f1_score": f1_05,
            "recall": recall_05,
            "precision": precision_05
        },
        "metrics_threshold_independent": {
            "auprc": auprc,
            "roc_auc": roc_auc,
            "lift_1_percent": lift_1pct,
            "tp_at_500": tp_at_500,
            "precision_at_500": precision_at_500
        },
        "feature_importance_summary": {
            "total_gain_tabular": float(total_gain_tabular),
            "total_gain_gnn": float(total_gain_gnn),
            "pct_tabular_gain": float(pct_tabular),
            "pct_gnn_gain": float(pct_gnn)
        },
        "runtime_seconds": {
            "data_preprocessing": data_prep_time,
            "gnn_training": gnn_train_time,
            "xgb_training": xgb_train_time,
            "total_test_inference": total_test_inf_time,
            "total_pipeline": data_prep_time + gnn_train_time + xgb_train_time + total_test_inf_time
        },
        "hyperparameters": {
            "hidden_channels": hidden_channels,
            "num_layers": num_layers,
            "aggr": aggr,
            "dropout": dropout,
            "gnn_epochs": epochs,
            "gnn_batch_size": batch_size,
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "best_gnn_epoch": best_epoch,
            "best_gnn_val_auprc": best_val_auprc,
            "num_neighbors": num_neighbors,
            "xgb_n_estimators": cls_cfg.get('n_estimators', 100),
            "xgb_max_depth": cls_cfg.get('max_depth', 4),
            "xgb_min_child_weight": cls_cfg.get('min_child_weight', 10),
            "xgb_scale_pos_weight": scale_pos_weight
        }
    }

    results_path = result_dir / f"exp21a_{sampling_method}_results.json"
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=4)
    print(f"\nHasil metrik tersimpan ke: {results_path}")

    # Simpan riwayat training
    history_path = result_dir / f"exp21a_{sampling_method}_history.json"
    with open(history_path, 'w') as f:
        json.dump(gnn_epoch_history, f, indent=4)

    # Simpan array prediksi kontinu untuk visualisasi
    pred_path = result_dir / f"exp21a_{sampling_method}_predictions.npz"
    np.savez_compressed(
        pred_path,
        test_probs=test_probs.astype(np.float32),
        test_labels=test_labels.astype(np.int8),
        val_probs=val_probs.astype(np.float32),
        val_labels=val_labels.astype(np.int8)
    )
    print(f"Array prediksi tersimpan ke: {pred_path}")

if __name__ == "__main__":
    main()



In [ ]:
%%writefile run_all.py
import os
import subprocess
import sys
import yaml
from pathlib import Path

def run_script(script_name, args=[]):
    print(f"\n{'='*70}")
    print(f"  >>> MENJALANKAN SCRIPT: {script_name} {' '.join(args)}")
    print(f"{'='*70}\n")
    
    command = [sys.executable, script_name] + args
    result = subprocess.run(command)
    
    if result.returncode != 0:
        print(f"\n[ERROR] Eksekusi {script_name} gagal dengan kode {result.returncode}!")
        sys.exit(result.returncode)
    else:
        print(f"\n[SUCCESS] {script_name} selesai dieksekusi.\n")

if __name__ == "__main__":
    print("="*70)
    print("  MEMULAI EKSPERIMEN 21A: ANTI-LEAKAGE & CLEAN ATTRIBUTE PIPELINE")
    print("="*70)
    
    # 1. Install dependencies jika di environment Kaggle/Colab
    if os.path.exists("/kaggle/input") or os.path.exists("/content"):
        print("\n[INFO] Menginstall dependencies (torch_geometric & pyg-lib)...")
        try:
            import torch
            pt_version = torch.__version__.split('+')[0]
            cuda_version = torch.version.cuda.replace('.', '') if torch.cuda.is_available() and torch.version.cuda else 'cpu'
            if cuda_version != 'cpu':
                url = f"https://data.pyg.org/whl/torch-{pt_version}+cu{cuda_version}.html"
                os.system(f"pip install torch_geometric pyg_lib torch_scatter torch_sparse -f {url} > /dev/null 2>&1")
            else:
                os.system("pip install torch_geometric > /dev/null 2>&1")
        except Exception as e:
            print(f"[WARNING] Fallback install torch_geometric: {e}")
            os.system("pip install torch_geometric > /dev/null 2>&1")

    # 2. Deteksi path konfigurasi
    if os.path.exists("/kaggle/input"):
        config_path = "config.kaggle.yaml"
        # Deteksi otomatis CSV di Kaggle
        csv_path = None
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'credit_card_transactions-ibm_v2.csv' in files:
                csv_path = os.path.join(root, 'credit_card_transactions-ibm_v2.csv')
                break
        if csv_path:
            print(f"[INFO] Dataset ditemukan di: {csv_path}")
            with open(config_path, 'r') as f:
                cfg = yaml.safe_load(f)
            cfg['data']['transactions'] = csv_path
            with open(config_path, 'w') as f:
                yaml.dump(cfg, f)
    elif os.path.exists("/content"):
        config_path = "config.colab.yaml"
    else:
        config_path = "config.reference.yaml"

    # 3. Jalankan eksperimen untuk metode sampling
    sampling_methods = ["uniform", "topology", "importance"]
    for method in sampling_methods:
        print(f"\n{'*'*70}")
        print(f"  >>> MULAI EXP21A DENGAN SAMPLING: {method.upper()}")
        print(f"{'*'*70}\n")
        run_script("main.py", ["--config", config_path, "--sampling", method])
        
    # 4. Generate Visualisasi Hasil (Chart Ilmiah)
    if os.path.exists("visualize_results.py"):
        print("\n" + "="*70)
        print("  >>> MENJALANKAN GENERASI VISUALISASI HASIL EXP21A")
        print("="*70 + "\n")
        try:
            with open(config_path, 'r') as f:
                cfg_loaded = yaml.safe_load(f)
            res_dir = cfg_loaded.get('output', {}).get('result_dir', 'result/exp21a')
            plots_dir = os.path.join(res_dir, 'plots')
            run_script("visualize_results.py", ["--result-dir", res_dir, "--output-dir", plots_dir])
        except Exception as e:
            print(f"[WARNING] Gagal menjalankan visualisasi otomatis: {e}")

    print("\n" + "="*70)
    print("  🎉 SELURUH PROSES PELATIHAN, EVALUASI, DAN AUDIT EXP21A SELESAI! 🎉")
    print("="*70)



In [ ]:
!python run_all.py